In [25]:
import numpy as np
import pandas as pd

### Importing Raw Datasets

In [26]:
raw_crime = pd.read_csv('datasets/raw/raw_crime_2024.csv', sep=';')
raw_nbh = pd.read_excel('datasets/raw/raw_nbh_2024.xlsx')

In [27]:
raw_nbh.shape

(18310, 126)

### Check how many 'BU' for Buurten

In [28]:
print("Crime Cols:", crime.columns)
print("NBH Cols:", nbh.columns)

Crime Cols: Index(['ID', 'SoortMisdrijf', 'WijkenEnBuurten', 'Perioden',
       'GeregistreerdeMisdrijven_1', 'Gemeentenaam_2', 'SoortRegio_3',
       'merge_key'],
      dtype='object')
NBH Cols: Index(['gwb_code_10', 'gwb_code_8', 'regio', 'gm_naam', 'recs', 'gwb_code',
       'ind_wbi', 'a_inw', 'a_man', 'a_vrouw',
       ...
       'g_afs_sc', 'g_3km_sc', 'a_opp_ha', 'a_lan_ha', 'a_wat_ha', 'pst_mvp',
       'pst_dekp', 'ste_mvs', 'ste_oad', 'merge_key'],
      dtype='object', length=127)


In [29]:
# Print counts of GM/WK/BU for crime dataset
print("GM_crime:", (raw_crime['WijkenEnBuurten'].astype(str).str.startswith('GM')).sum())
print("WK_crime:", (raw_crime['WijkenEnBuurten'].astype(str).str.startswith('WK')).sum())
print("BU_crime:", (raw_crime['WijkenEnBuurten'].astype(str).str.startswith('BU')).sum())
print("Total Crime Rows", raw_crime.shape[0])
print()  # Blank line for readability

# Print counts of GM/WK/BU for neighbourhood dataset
print("GM_nbh:", (raw_nbh['gwb_code_10'].astype(str).str.startswith('GM')).sum())
print("WK_nbh:", (raw_nbh['gwb_code_10'].astype(str).str.startswith('WK')).sum())
print("BU_nbh:", (raw_nbh['gwb_code_10'].astype(str).str.startswith('BU')).sum())
print("Total NBH Rows", raw_nbh.shape[0])


GM_crime: 343
WK_crime: 3424
BU_crime: 14730
Total Crime Rows 18498

GM_nbh: 342
WK_nbh: 3393
BU_nbh: 14574
Total NBH Rows 18310


### Remove 'GM' and 'WK' from 'WijkenEnBuurten' and 'gwb_code_10'

In [30]:
# Keep only neighborhood (BU) level: drop rows where code starts with 'GM' or 'WK'.
crime = raw_crime[~raw_crime['WijkenEnBuurten'].astype(str).str.strip().str.startswith(('GM', 'WK'), na=False)]
nbh = raw_nbh[~raw_nbh['gwb_code_10'].astype(str).str.strip().str.startswith(('GM', 'WK'), na=False)]


### Remove one remaining row which starts with 'NL'

In [31]:
# drop rows where code starts with 'NL'
crime = crime[~crime['WijkenEnBuurten'].astype(str).str.strip().str.startswith(('NL'), na=False)]
nbh = nbh[~nbh['gwb_code_10'].astype(str).str.strip().str.startswith(('NL'), na=False)]

print("Rows in crime after dropping 'GM', 'WK', and 'NL':", crime.shape[0])
print("Rows in nbh after dropping 'GM', 'WK', and 'NL':", nbh.shape[0])

Rows in crime after dropping 'GM', 'WK', and 'NL': 14730
Rows in nbh after dropping 'GM', 'WK', and 'NL': 14574


## Merging datasets

### Checking missing values for merge column - 0

In [32]:
crime["WijkenEnBuurten"].isna().mean(), nbh["gwb_code_10"].isna().mean()

(0.0, 0.0)

### Standardize & Create join keys

In [33]:
# Standardize join keys (strip whitespace, uppercase, handle missing tokens).

def clean_key(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
        .str.upper()
        .replace({"NAN": np.nan, "NONE": np.nan})
    )

crime = crime.copy()
nbh = nbh.copy()

crime["merge_key"] = clean_key(crime["WijkenEnBuurten"])
nbh["merge_key"] = clean_key(nbh["gwb_code_10"])


### Check for Duplicates

In [34]:
crime_key_dupes = crime["merge_key"].duplicated(keep=False).mean()
nbh_key_dupes   = nbh["merge_key"].duplicated(keep=False).mean()

crime_key_dupes, nbh_key_dupes

(0.0, 0.0)

In [35]:
crime["merge_key"].value_counts().head(10)
nbh["merge_key"].value_counts().head(10)

merge_key
BU00140000    1
BU09831307    1
BU09810005    1
BU09810006    1
BU09810007    1
BU09810008    1
BU09810009    1
BU09810100    1
BU09831101    1
BU09831102    1
Name: count, dtype: int64

### Merging

In [36]:
merged = nbh.merge(
    crime.drop(columns=["WijkenEnBuurten"], errors="ignore"),
    on="merge_key",
    how="left",
    validate="one_to_one"
)

merged.shape

(14574, 133)

### Check merging percentage - 99.45%

In [37]:
match_rate = merged["GeregistreerdeMisdrijven_1"].notna().mean()
match_rate

0.9945107726087553

### Remove unmerged rows - 80

In [38]:
unmatched = merged.loc[merged["GeregistreerdeMisdrijven_1"].isna(), "merge_key"]
unmatched.head(), unmatched.nunique()

# Remove the unmerged rows
merged = merged[merged["GeregistreerdeMisdrijven_1"].notna()].reset_index(drop=True)

In [39]:
merged.shape

(14494, 133)

### Merge neighborhood geometry (GeoPackage)

Join official CBS / PDOK **wijkenbuurten** polygons on the same `merge_key` as above (`buurtcode` standardized ↔ `gwb_code_10`). The CSV export in the next section stays tabular-only; geometry is written to a separate GeoPackage for downstream use (e.g. notebook 4 pre-processing).

In [40]:
import geopandas as gpd

# --- Edit paths if your file lives elsewhere ---
GPKG_PATH = "datasets/raw/wijkenbuurten_2024.gpkg"
GPKG_LAYER = "buurten"  # layer name when the file has multiple layers (e.g. buurten, wijken, gemeenten)
GEO_KEY_COL = "buurtcode"

geo = gpd.read_file(GPKG_PATH, layer=GPKG_LAYER)
geo_small = geo[[GEO_KEY_COL, "geometry"]].copy()
geo_small["merge_key"] = clean_key(geo_small[GEO_KEY_COL])

merged_geom = merged.merge(
    geo_small[["merge_key", "geometry"]],
    on="merge_key",
    how="left",
)
merged_gdf = gpd.GeoDataFrame(merged_geom, geometry="geometry", crs=geo.crs)

n = len(merged_gdf)
matched = merged_gdf.geometry.notna().sum()
print(f"Geometry rows matched: {matched} / {n} ({100 * matched / n:.2f}%)")

Geometry rows matched: 14494 / 14494 (100.00%)


### Export final merged dataset

In [41]:
from pathlib import Path

PREPROC_DIR = Path("datasets/pre-processing")
PREPROC_DIR.mkdir(parents=True, exist_ok=True)

final = merged.copy()

out_path = PREPROC_DIR / "merged_crime_nbh_2024.csv"
final.to_csv(out_path, index=False)
str(out_path)

'datasets/pre-processing/merged_crime_nbh_2024.csv'

### Export merged dataset with geometry (GeoPackage)

Notebook 4 attaches these polygons via this file instead of merging the CBS source `.gpkg` again.

In [42]:
from pathlib import Path

PREPROC_DIR = Path("datasets/pre-processing")
PREPROC_DIR.mkdir(parents=True, exist_ok=True)

gpkg_out = PREPROC_DIR / "merged_crime_nbh_2024.gpkg"
merged_gdf.to_file(gpkg_out, driver="GPKG")
str(gpkg_out)

'datasets/pre-processing/merged_crime_nbh_2024.gpkg'